In [2]:
# 导入必要的库
import akshare as ak
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print("akshare 版本:", ak.__version__)

akshare 版本: 1.18.21


## 步骤0: 获取可用的美股代码列表

**重要提示**: 根据 akshare 文档，股票代码应该通过 `stock_us_spot_em()` 函数获取。这个函数返回所有可用的美股代码列表。

In [8]:
# 获取所有美股的实时行情数据，从中可以获取股票代码
# 参考文档: https://akshare.akfamily.xyz/data/stock/stock.html#id2

import os
from datetime import datetime

# 定义保存路径
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)
csv_file = os.path.join(data_dir, "stock_us_spot_em.csv")
pickle_file = os.path.join(data_dir, "stock_us_spot_em.pkl")

# 尝试从本地加载
stock_us_spot_df = None
if os.path.exists(csv_file):
    try:
        print(f"从本地加载数据: {csv_file}")
        stock_us_spot_df = pd.read_csv(csv_file)
        file_time = datetime.fromtimestamp(os.path.getmtime(csv_file))
        print(f"✓ 成功加载 {len(stock_us_spot_df)} 只股票代码 (保存时间: {file_time.strftime('%Y-%m-%d %H:%M:%S')})")
        print("提示: 如需更新数据，请删除文件后重新运行")
    except Exception as e:
        print(f"⚠ 加载本地数据失败: {e}，将从网络获取")

# 如果本地没有数据，从网络获取
if stock_us_spot_df is None:
    try:
        print("\n正在从网络获取美股代码列表...")
        
        # 修复 tqdm/ipywidgets 错误：Monkey patch tqdm.notebook 模块
        # 问题：akshare 内部使用 tqdm，在 notebook 环境中会尝试使用 ipywidgets
        # 解决：在调用前临时替换 tqdm.notebook.tqdm 为标准 tqdm
        
        try:
            import tqdm
            from tqdm import tqdm as tqdm_std
            import tqdm.notebook as tqdm_nb
            
            # 保存原始函数
            original_tqdm_nb = tqdm_nb.tqdm if hasattr(tqdm_nb, 'tqdm') else None
            original_status_printer = tqdm_nb.status_printer if hasattr(tqdm_nb, 'status_printer') else None
            
            # 创建一个安全的 status_printer，避免 ipywidgets 依赖
            def safe_status_printer(file, total, desc=None, ncols=None, **kwargs):
                # 返回一个简单的对象，模拟 tqdm 的容器
                class FakeContainer:
                    def __init__(self):
                        self.bar_style = ''
                    def __call__(self, *args, **kwargs):
                        return self
                    def update(self, *args, **kwargs):
                        pass
                    def close(self, *args, **kwargs):
                        pass
                return FakeContainer()
            
            # 替换 notebook.tqdm 为标准 tqdm
            tqdm_nb.tqdm = tqdm_std
            tqdm_nb.status_printer = safe_status_printer
            
            print("  ✓ 已修复 tqdm notebook 模式问题")
        except Exception as patch_error:
            print(f"  ⚠ tqdm patch 警告: {patch_error}")
            print("  提示: 如果仍然失败，请安装 ipywidgets: uv pip install ipywidgets")
        
        # 尝试获取数据
        stock_us_spot_df = ak.stock_us_spot_em()
        
        # 恢复原始设置（可选，通常不需要）
        # try:
        #     if original_tqdm_nb is not None:
        #         tqdm_nb.tqdm = original_tqdm_nb
        #     if original_status_printer is not None:
        #         tqdm_nb.status_printer = original_status_printer
        # except:
        #     pass
        
        print(f"✓ 成功获取 {len(stock_us_spot_df)} 只美股代码")
        
        # 保存到本地
        print(f"\n正在保存到本地...")
        stock_us_spot_df.to_csv(csv_file, index=False, encoding='utf-8-sig')
        stock_us_spot_df.to_pickle(pickle_file)
        print(f"✓ 已保存到:")
        print(f"  - CSV格式: {csv_file}")
        print(f"  - Pickle格式: {pickle_file}")
    except Exception as e:
        import traceback
        error_msg = str(e)
        print(f"\n❌ 获取美股代码列表失败: {error_msg}")
        
        # 检查是否是 tqdm/ipywidgets 错误
        if 'ipywidgets' in error_msg.lower() or 'iprogress' in error_msg.lower() or 'tqdm' in error_msg.lower():
            print("\n" + "="*60)
            print("🔧 解决方案（tqdm/ipywidgets 错误）:")
            print("="*60)
            print("方法1（推荐）: 安装 ipywidgets")
            print("  运行命令: uv pip install ipywidgets")
            print("  然后重新运行此 cell")
            print("\n方法2: 使用本地数据")
            print("  如果之前已经成功下载过数据，删除 data/stock_us_spot_em.csv")
            print("  然后重新运行，会从本地加载")
            print("\n方法3: 手动下载数据")
            print("  在 Python 脚本中运行（非 notebook 环境）:")
            print("  import akshare as ak")
            print("  df = ak.stock_us_spot_em()")
            print("  df.to_csv('data/stock_us_spot_em.csv', index=False, encoding='utf-8-sig')")
        else:
            print("\n详细错误信息:")
            traceback.print_exc()
            print("\n提示: 可以使用 stock_us_spot_em() 函数获取所有可用的美股代码")
        
        stock_us_spot_df = None

# 显示数据信息
if stock_us_spot_df is not None:
    print(f"\n{'='*60}")
    print(f"数据概览:")
    print(f"{'='*60}")
    print(f"列名: {list(stock_us_spot_df.columns)}")
    print("\n前10只股票:")
    display(stock_us_spot_df.head(10))
    
    # 查找一些知名公司的代码
    print("\n查找知名公司代码:")
    famous_companies = {
        '苹果': ['Apple', 'AAPL'],
        '微软': ['Microsoft', 'MSFT'],
        '谷歌': ['Google', 'GOOG', 'GOOGL'],
        '亚马逊': ['Amazon', 'AMZN'],
        '特斯拉': ['Tesla', 'TSLA'],
        'Meta': ['Meta', 'FB', 'META'],
        '英伟达': ['NVIDIA', 'NVDA'],
        '阿里巴巴': ['Alibaba', 'BABA']
    }
    
    print("\n" + "="*60)
    for company_name, keywords in famous_companies.items():
        matches = stock_us_spot_df[
            stock_us_spot_df['名称'].str.contains('|'.join(keywords), case=False, na=False) |
            stock_us_spot_df['代码'].str.contains('|'.join(keywords), case=False, na=False)
        ]
        if not matches.empty:
            print(f"\n{company_name}:")
            display(matches[['代码', '名称', '最新价']].head(3))
    
    # 使用说明
    print("\n" + "="*60)
    print("使用说明:")
    print("  1. 使用 stock_us_spot_df['代码'] 可以获取所有股票代码列表")
    print("  2. 代码示例: codes = stock_us_spot_df['代码'].tolist()")
    print("  3. 然后使用这些代码调用 stock_us_hist()")
    print(f"  4. 数据已保存到本地，下次可以直接加载")
    print(f"  5. 如需更新数据，删除 {csv_file} 后重新运行")
    
    # 创建 README 文件说明 data 目录
    readme_file = os.path.join(data_dir, "README.md")
    if not os.path.exists(readme_file):
        with open(readme_file, 'w', encoding='utf-8') as f:
            f.write("# 数据目录\n\n")
            f.write("此目录用于保存从 akshare 获取的美股代码列表数据。\n\n")
            f.write("## 文件说明\n\n")
            f.write("- `stock_us_spot_em.csv`: CSV 格式的美股代码列表（UTF-8编码）\n")
            f.write("- `stock_us_spot_em.pkl`: Pickle 格式的美股代码列表（保留数据类型）\n\n")
            f.write("## 使用说明\n\n")
            f.write("数据会在首次运行时自动下载并保存。\n")
            f.write("后续运行时会自动从本地加载，避免重复请求网络。\n")
            f.write("如需更新数据，删除对应的文件后重新运行代码即可。\n")
        print(f"\n✓ 已创建说明文件: {readme_file}")


正在从网络获取美股代码列表...
提示: 如果遇到 tqdm/ipywidgets 错误，可以:
  1. 安装 ipywidgets: uv pip install ipywidgets
  2. 或者使用已有的本地数据文件


Traceback (most recent call last):
  File "/var/folders/c6/jk_lx_9x3zlfx0vp5g4rpcvm0000gp/T/ipykernel_26439/1901860493.py", line 34, in <module>
    stock_us_spot_df = ak.stock_us_spot_em()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/frank/wk/github/quant_in_action/.venv/lib/python3.12/site-packages/akshare/stock_feature/stock_hist_em.py", line 1614, in stock_us_spot_em
    temp_df = fetch_paginated_data(url, params)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/frank/wk/github/quant_in_action/.venv/lib/python3.12/site-packages/akshare/utils/func.py", line 46, in fetch_paginated_data
    for page in tqdm(range(2, total_page + 1), leave=False):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/frank/wk/github/quant_in_action/.venv/lib/python3.12/site-packages/tqdm/notebook.py", line 234, in __init__
    self.container = self.status_printer(self.fp, total, self.desc, self.ncols)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

获取美股代码列表失败: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

详细错误信息:

提示: 可以使用 stock_us_spot_em() 函数获取所有可用的美股代码


In [ ]:
# 方法1: 先从 stock_us_spot_em() 获取代码，然后使用该代码获取历史数据
# 这是推荐的方法，因为可以确保代码格式正确

try:
    # 步骤1: 获取股票代码列表（优先使用本地保存的数据）
    print("步骤1: 获取股票代码列表...")
    data_dir = "data"
    csv_file = os.path.join(data_dir, "stock_us_spot_em.csv")
    
    if os.path.exists(csv_file):
        print("  从本地加载数据...")
        stock_us_spot_df = pd.read_csv(csv_file)
        print(f"  ✓ 从本地加载 {len(stock_us_spot_df)} 只股票代码")
    else:
        print("  从网络获取数据...")
        stock_us_spot_df = ak.stock_us_spot_em()
        print(f"  ✓ 从网络获取 {len(stock_us_spot_df)} 只股票代码")
    
    # 步骤2: 查找苹果公司的代码
    print("\n步骤2: 查找苹果公司(AAPL)的代码...")
    apple_matches = stock_us_spot_df[
        stock_us_spot_df['名称'].str.contains('Apple', case=False, na=False) |
        stock_us_spot_df['代码'].str.contains('AAPL', case=False, na=False)
    ]
    
    if apple_matches.empty:
        print("未找到苹果公司，尝试使用常见代码格式...")
        # 尝试常见的代码格式
        test_symbols = ["AAPL", "105.AAPL"]
    else:
        # 使用 stock_us_spot_em() 返回的代码
        apple_code = apple_matches.iloc[0]['代码']
        print(f"找到苹果公司代码: {apple_code}")
        test_symbols = [apple_code, "AAPL", "105.AAPL"]  # 也尝试其他格式
    
    # 步骤3: 使用获取到的代码来获取历史数据
    print("\n步骤3: 获取历史数据...")
    stock_us_hist_df = None
    successful_symbol = None
    
    for symbol in test_symbols:
        try:
            print(f"  尝试代码: {symbol}")
            stock_us_hist_df = ak.stock_us_hist(
                symbol=symbol,
                period="daily",
                start_date="20240101",
                end_date="20241231",
                adjust=""  # 先尝试不复权
            )
            
            if stock_us_hist_df is not None and not stock_us_hist_df.empty:
                successful_symbol = symbol
                print(f"  ✓ 成功！")
                break
            elif stock_us_hist_df is not None and stock_us_hist_df.empty:
                print(f"  ⚠ 返回空数据")
            else:
                print(f"  ✗ 返回 None")
        except Exception as e:
            print(f"  ✗ 错误: {str(e)[:80]}")
            continue
    
    # 步骤4: 显示结果
    if stock_us_hist_df is not None and not stock_us_hist_df.empty:
        print(f"\n{'='*60}")
        print(f"✓ 成功获取数据 (使用代码: {successful_symbol})")
        print(f"{'='*60}")
        print(f"数据形状: {stock_us_hist_df.shape}")
        print(f"\n列名: {list(stock_us_hist_df.columns)}")
        print("\n前10条数据:")
        display(stock_us_hist_df.head(10))
        print("\n数据基本信息:")
        print(stock_us_hist_df.info())
    else:
        print(f"\n{'='*60}")
        print("❌ 所有代码格式都失败了")
        print(f"{'='*60}")
        print("建议：")
        print("1. 检查网络连接")
        print("2. 确认 stock_us_spot_em() 返回的代码格式")
        print("3. 尝试使用更近的日期范围")
        print("4. 更新 akshare: uv pip install akshare --upgrade")
    
except Exception as e:
    import traceback
    print(f"获取数据时出错: {e}")
    print("\n详细错误信息:")
    traceback.print_exc()

尝试获取未复权数据...
获取数据时出错: 'NoneType' object is not subscriptable

详细错误信息:

请检查网络连接和股票代码是否正确


Traceback (most recent call last):
  File "/var/folders/c6/jk_lx_9x3zlfx0vp5g4rpcvm0000gp/T/ipykernel_26439/446202153.py", line 12, in <module>
    stock_us_hist_df = ak.stock_us_hist(
                       ^^^^^^^^^^^^^^^^^
  File "/Users/frank/wk/github/quant_in_action/.venv/lib/python3.12/site-packages/akshare/stock_feature/stock_hist_em.py", line 1725, in stock_us_hist
    if not data_json["data"]["klines"]:
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^
TypeError: 'NoneType' object is not subscriptable
